# Project status

This file will be showing the status of this project. I think that in terms of code and functionality this codebase is quite beautiful. However, in disease prediction it sucks! There is some disparity between how my models use the graph structure. They don't. 

In [1]:
import importlib.util
import sys
from datetime import date
import pandas as pd
import numpy as np
from typing import Dict, Literal, Optional
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
import os
from pathlib import Path

wissdaten_dir    = os.environ.get('TMPDIR') +'/wissdaten/'
data_env         = os.path.join(wissdaten_dir, 'ZKI-PH4/deschrijvers_wissdaten/data')

data_module      = module_path = data_env + "/__init__.py"
spec = importlib.util.spec_from_file_location("data", module_path)
module = importlib.util.module_from_spec(spec)
sys.modules["data"] = module
spec.loader.exec_module(module)

from data.visuals import *
colors_dict = {}
colors_dict['context']    = hexcodes['sky blue']
colors_dict['future']     = hexcodes['soft red']
colors_dict['predictions']= hexcodes['moss green']

In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from torch_geometric.data import Data
import os
import sys

from tests import *

from src.dataloading.gnndataloader import GNNDataLoader
from src.dataloading.epidataloader import EpiDataLoader
from src.models.temporal_gcn import TemporalGCNModel
from src.models.spatial_gcn import SpatialGCNModel
from src.models.node_lstm import NodeLSTM

n_periods = 8
n_epochs  = 50
horizon   = 1
lags      = range(4,5)


epidata = EpiDataLoader('influenza', data_env, aggr_level= '03', min_date='2012-06-01',max_date='2020-06-01')
# epidata.add_time_features()
epidata.log_transform_target()
epidata.normalize('2018-06-01','2019-06-01','zscore')
epidata.add_lagged_features(lags = lags)

gnnloader1 = GNNDataLoader(epidata).retrieve_graph('identity_graph').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)
gnnloader2 = GNNDataLoader(epidata).retrieve_graph('boolean_neighbors').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)
gnnloader3 = GNNDataLoader(epidata).retrieve_graph('gravity_model_log').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)
gnnloader4 = GNNDataLoader(epidata).retrieve_graph('gravity_model_top8_log').construct_dataloaders(periods=n_periods, prediction_horizon=horizon)

dataloaders = {'identity_graph': gnnloader1,
                    'boolean_neighbors': gnnloader2,
                    'gravity_model1': gnnloader3,
                    'gravity_model2': gnnloader4}

models = {}

model_classes    = {'tgcn': TemporalGCNModel,
                    'sgcn': SpatialGCNModel}

for modeltype, cl in model_classes.items():

    for dataloader_name, dataloader in dataloaders.items():

        graph_validation = GraphValidator(dataloader, graphname = dataloader_name)
        graph_validation.run_tests()

        name = modeltype + "_" + dataloader_name 

        ml = cl(name = name, dataloader = dataloader)
        ml.set_model_hparams()
        ml.set_global_hparams(lr = 0.00005, n_epochs=n_epochs, scheduler_kwargs={'step_size':15}, min_delta = 0.1)
        ml.train(show_loss = False)
        ml.forecast()
        # ml.show_forecasts(norm = True)

        models[name] = ml

        validator = GraphUsageValidator(ml)
        validator.run_tests()

/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/de-schrijvers/.conda/envs/gnenv/lib/python3.11/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


logs saved
Dataloader Snapshot: Data(x=[411, 2, 8], edge_index=[2, 411], y=[411, 1], edge_weight=[411])
Epoch 1 train loss: 5.1684, val loss: 9.9530 ✓ (new best)
Epoch 2 train loss: 5.0990, val loss: 9.6222 ✓ (new best)
Epoch 3 train loss: 5.0405, val loss: 9.3143 ✓ (new best)
Epoch 4 train loss: 4.9981, val loss: 9.0644 ✓ (new best)
Epoch 5 train loss: 4.9548, val loss: 8.8555 ✓ (new best)
Epoch 6 train loss: 4.9111, val loss: 8.6541 ✓ (new best)
Epoch 7 train loss: 4.8501, val loss: 8.4208 ✓ (new best)
Epoch 8 train loss: 4.7677, val loss: 8.1263 ✓ (new best)
Epoch 9 train loss: 4.6541, val loss: 7.7550 ✓ (new best)
Epoch 10 train loss: 4.5203, val loss: 7.3402 ✓ (new best)
Epoch 11 train loss: 4.3731, val loss: 6.9215 ✓ (new best)
Epoch 12 train loss: 4.2427, val loss: 6.5516 ✓ (new best)
Epoch 13 train loss: 4.1282, val loss: 6.2344 ✓ (new best)
Epoch 14 train loss: 4.0424, val loss: 5.9697 ✓ (new best)
Epoch 15 train loss: 3.8789, val loss: 5.9792 (patience: 1/15)
Epoch 16 train l